<a href="https://colab.research.google.com/github/harinijk/NewsVsClickbait/blob/main/csci4521HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget "https://raw.githubusercontent.com/SJGuy-UMN/CSCI4521/refs/heads/main/clickbait_data.csv"

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import nltk.stem
from wordcloud import WordCloud
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import plotly.express as px
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

Question 1

In [ ]:
data = pd.read_csv("clickbait_data.csv")
print(data.head())
len(data)

In [ ]:
# 2 data subsampling
data = data.sample(20000, random_state=50)
# googled to find out how to reorder indexes after sampling which was required later
data = data.reset_index(drop=True)
posts = data["headline"]
labels = data["clickbait"]

In [ ]:
# 1 vectorization, #3 feature extraction
english_stemmer = nltk.stem.SnowballStemmer('english')
class StemmedTfidfVectorizer(TfidfVectorizer):
   def build_analyzer(self):
     analyzer = super(StemmedTfidfVectorizer, self).build_analyzer()
     return lambda doc: (english_stemmer.stem(w) for w in analyzer(doc))

vectorizer = StemmedTfidfVectorizer(min_df=10, max_df=0.8,stop_words='english')
X_train = vectorizer.fit_transform(posts)

In [ ]:
# 4 statistics
print("size of training data after subsampling is ", X_train.shape[0])
print("the number of unique words (features) is ", X_train.shape[1])
print("number of samples with each label is ", labels.value_counts())


Question 2

In [ ]:
#i)
km = KMeans(n_clusters=2, random_state=100)
clusters = km.fit_predict(X_train)
data["cluster"] = clusters

#ii)
cluster_0 = data[data["cluster"] == 0]
clickbait_0 = cluster_0["clickbait"].sum()
percent_0 = cluster_0["clickbait"].mean() * 100
print("Cluster 0")
print("  total number of items", len(cluster_0))
print("  number of clickbait items", clickbait_0)
print("  clickbait percentage", percent_0)
print()

cluster_1 = data[data["cluster"] == 1]
clickbait_1 = cluster_1["clickbait"].sum()
percent_1 = cluster_1["clickbait"].mean() * 100
print("Cluster 1")
print("  total number of items", len(cluster_1))
print("  number of clickbait items", clickbait_1)
print("  clickbait percentage", percent_1)

Question 3

In [ ]:
# i)

km = KMeans(n_clusters=7, random_state=100)
km.fit(X_train)
clusters = km.labels_
data["cluster_multi"] = clusters


In [ ]:
#ii)
# wordcloud method based on lecture
feature_names = vectorizer.get_feature_names_out()
X_new = X_train.toarray()

for cluster_id in range(7):
    matching_indices = (clusters == cluster_id)
    tfidf_sum = X_new[matching_indices].sum(axis=0)
    word_scores = {}

    for i in range(len(feature_names)):
      word = feature_names[i]
      score = tfidf_sum[i]

      if score > 0:
        word_scores[word] = score

    word_cloud = WordCloud(background_color='white',max_words=60).generate_from_frequencies(word_scores)

    plt.figure()
    plt.imshow(word_cloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Cluster {cluster_id}")
    plt.show()

In [ ]:
#iii)

distances = cdist(X_train.toarray(), km.cluster_centers_)
headlines = data["headline"].tolist()

for cluster_id in range(7):
    print(f"\nCluster {cluster_id} 3 closest Headlines:")
    cluster_indices = np.where(clusters == cluster_id)[0]
    cluster_distances = distances[cluster_indices, cluster_id]
    closest_indices = cluster_indices[np.argsort(cluster_distances)[:3]]

    for idx in closest_indices:
        print(headlines[idx])

Question 4

In [ ]:
#fidning cluster type for next part

clusters = {}

for id in range(7):

    bool = (data["cluster_multi"] ==id)
    cluster_new = data[bool]
    clickbait_percent = cluster_new["clickbait"].mean()

    if clickbait_percent >= 0.5:
       clusters[id] = "clickbait"
    else:
      clusters[id] = "news"

    print("Cluster", id, "Type is", clusters[id])
    print()

In [ ]:
# correct news in clickbait

for cid in clusters:
    if clusters[cid] == "clickbait":
        for i in range(len(data)):
            if data["cluster_multi"][i] == cid and data["clickbait"][i] == 0:
                print("News inside Clickbait Cluster", cid)
                print(data["headline"][i])
                break

In [ ]:
# clickbait in correct news

for cid in clusters:
    if clusters[cid] == "news":
        for i in range(len(data)):
            if data["cluster_multi"][i] == cid and data["clickbait"][i] == 1:
                print("Clickbait inside News Cluster", cid)
                print(data["headline"][i])
                break

In [ ]:
#Prompted chatgpt for examples of clickbait/legitimate news headlines
# 1st and 2nd are clickbait; 2nd and 4th are legitimate

headlines = ["You won't Believe What This CEO Did After Losing $10 Million!", "Doctors Hate Her for This One Simple Trick!", "Federal Reserve Raises Interest Rates by 0.25%", "Supreme Court Rules on Student Loan Forgiveness Plan"]
vectorize = vectorizer.transform(headlines)

#cluster I expect:

distances = cdist(vectorize.toarray(), km.cluster_centers_)

for i in range(len(headlines)):
    print(headlines[i])

    for cid in range(len(km.cluster_centers_)):
        print("Cluster distance", cid , distances[i][cid])

    closest = np.argmin(distances[i])
    print("closest cluster is", closest)
    print("the cluster type is", clusters[closest])


Question 5

In [ ]:
doc_features = X_train.toarray()
labels = data["clickbait"].values
ids = np.arange(0, len(labels))
pca = PCA(n_components=3)
pc = pca.fit_transform(doc_features)

In [ ]:
pc_df = pd.DataFrame(data=pc, columns=['PC1','PC2','PC3'])
pc_df['Cluster'] = labels
pc_df['ID'] = ids
pc_df['Headline'] = data["headline"].values

pc_df.head()

In [ ]:
fig = px.scatter_3d(
    pc_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Cluster",
    hover_data=["ID","Headline"],
    title="PCA Projection"
)

fig.show()

In [ ]:
#2 1883 and 4148 far from cluster 0
print(data.iloc[1883]["headline"])
print(data.iloc[4148]["headline"])

In [ ]:
total_variance = np.sum(pca.explained_variance_ratio_)
print("total variance", total_variance*100)
print("PC1 explains:", pca.explained_variance_ratio_[0] * 100, "%")
print("PC2 explains:", pca.explained_variance_ratio_[1] * 100, "%")
print("PC3 explains:", pca.explained_variance_ratio_[2] * 100, "%")

In [ ]:
pca_full = PCA()
pca_full.fit(doc_features)

exp_var_cumul = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(8,5))
plt.plot(exp_var_cumul)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance Plot")
plt.show()

components_50 = np.argmax(exp_var_cumul > 0.5) + 1


In [ ]:
print("Number of components needed to capture > 50% variance is", components_50)

In [ ]:
#Extra Credit
X_new = X_train.toarray()
X_train, X_test, y_train, y_test = train_test_split(X_new,labels,test_size=0.2,random_state=1,stratify=labels)
pca = PCA(n_components=400, random_state=1)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)

y_train_pred = knn.predict(X_train)
y_test_pred = knn.predict(X_test)

print("Training Accuracy is", (accuracy_score(y_train, y_train_pred))*100,"%")
print("Testing Accuracy is", (accuracy_score(y_test, y_test_pred))*100,"%")
print("Training F1 Score is", (f1_score(y_train, y_train_pred))*100, "%")
print("Testing F1 Score is", (f1_score(y_test, y_test_pred)*100), "%")
